# BraTS Dataset - Empty .nii Files Checker and Remover

This notebook scans all patient ID folders, checks for empty .nii files, and removes them.

## 1. Import Required Libraries

In [19]:
import os
import glob
from pathlib import Path
import nibabel as nib
import numpy as np
from datetime import datetime

# For summary statistics
from collections import defaultdict

## 2. Define the Root Folder Path

In [20]:
# Define the root directory containing all patient folders
ROOT_DIR = r"F:\BraTs 2024 Data Unpatched\Training-Data"

print(f"Root directory: {ROOT_DIR}")
print(f"Directory exists: {os.path.exists(ROOT_DIR)}")

Root directory: F:\BraTs 2024 Data Unpatched\Training-Data
Directory exists: True


## 3. Function to Check if .nii File is Empty

In [21]:
def is_nii_file_empty(file_path):
    """
    Check if a .nii file is empty or corrupted.
    
    Parameters:
    -----------
    file_path : str
        Path to the .nii file
        
    Returns:
    --------
    tuple: (is_empty: bool, reason: str)
    """
    try:
        # Check if file size is 0
        file_size = os.path.getsize(file_path)
        if file_size == 0:
            return True, "File size is 0 bytes"
        
        # Try to load the file with nibabel
        img = nib.load(file_path)
        data = img.get_fdata()
        
        # Check if data array is empty or has zero size
        if data.size == 0:
            return True, "Data array is empty"
        
        # Check if all values are zero (might indicate empty/corrupted file)
        if np.all(data == 0):
            return True, "All voxel values are zero"
        
        # File appears to be valid
        return False, "File is valid"
        
    except Exception as e:
        # If we can't load the file, consider it corrupted/empty
        return True, f"Error loading file: {str(e)}"

# Test the function
print("Function defined successfully!")

Function defined successfully!


## 4. Scan Patient Folders and Find All .nii Files

In [22]:
# Get all patient ID folders (folders starting with BraTS-GLI-)
patient_folders = []
for item in os.listdir(ROOT_DIR):
    item_path = os.path.join(ROOT_DIR, item)
    if os.path.isdir(item_path) and item.startswith('BraTS-GLI-'):
        patient_folders.append(item_path)

patient_folders.sort()
print(f"Found {len(patient_folders)} patient folders")
print(f"\nFirst 5 patient folders:")
for folder in patient_folders[:5]:
    print(f"  - {os.path.basename(folder)}")

Found 419 patient folders

First 5 patient folders:
  - BraTS-GLI-02225-101
  - BraTS-GLI-02226-100
  - BraTS-GLI-02226-101
  - BraTS-GLI-02227-100
  - BraTS-GLI-02227-101


In [ ]:
# Collect all .nii files from all patient folders
all_nii_files = []
for folder in patient_folders:
    # Find all .nii and .nii.gz files in the folder
    nii_files = glob.glob(os.path.join(folder, "*.nii"))
    nii_gz_files = glob.glob(os.path.join(folder, "*.nii.gz"))
    all_nii_files.extend(nii_files)
    all_nii_files.extend(nii_gz_files)

print(f"Total .nii files found: {len(all_nii_files)}")
print(f"\nSample files:")
for file in all_nii_files[:5]:
    print(f"  - {os.path.basename(file)}")

Total .nii files found: 2095

Sample files:
  - BraTS-GLI-02225-101-seg.nii
  - BraTS-GLI-02225-101-t1c.nii
  - BraTS-GLI-02225-101-t1n.nii
  - BraTS-GLI-02225-101-t2f.nii
  - BraTS-GLI-02225-101-t2w.nii


: 

## 5. Check for Empty .nii Files

In [ ]:
# Check each file and identify empty ones
empty_files = []
valid_files = []
file_status = []

print("Checking files for emptiness...\n")
print("=" * 80)

for idx, file_path in enumerate(all_nii_files, 1):
    is_empty, reason = is_nii_file_empty(file_path)
    
    file_info = {
        'path': file_path,
        'filename': os.path.basename(file_path),
        'folder': os.pa5th.basename(os.path.dirname(file_path)),
        'size': os.path.getsize(file_path),
        'is_empty': is_empty,
        'reason': reason
    }
    
    file_status.append(file_info)
    
    if is_empty:
        empty_files.append(file_info)
        print(f"[{idx}/{len(all_nii_files)}] EMPTY: {file_info['folder']}/{file_info['filename']}")
        print(f"         Reason: {reason}")
        print(f"         Size: {file_info['size']} bytes")
        print()
    else:
        valid_files.append(file_info)
    
    # Progress indicator every 50 files
    if idx % 50 == 0:
        print(f"Progress: {idx}/{len(all_nii_files)} files checked...")

print("=" * 80)
print(f"\nSummary:")
print(f"  Total files scanned: {len(all_nii_files)}")
print(f"  Valid files: {len(valid_files)}")
print(f"  Empty/Corrupted files: {len(empty_files)}")

Checking files for emptiness...



Progress: 50/2095 files checked...
Progress: 100/2095 files checked...
Progress: 150/2095 files checked...
Progress: 200/2095 files checked...
Progress: 250/2095 files checked...
Progress: 300/2095 files checked...
Progress: 350/2095 files checked...
Progress: 400/2095 files checked...
Progress: 450/2095 files checked...
Progress: 500/2095 files checked...
Progress: 550/2095 files checked...
Progress: 600/2095 files checked...
[628/2095] EMPTY: BraTS-GLI-02336-100/BraTS-GLI-02336-100-t1n.nii
         Reason: Error loading file: [WinError 8] Not enough memory resources are available to process this command
         Size: 16777216 bytes

Progress: 650/2095 files checked...
Progress: 700/2095 files checked...
Progress: 750/2095 files checked...
Progress: 800/2095 files checked...
Progress: 850/2095 files checked...
Progress: 900/2095 files checked...
Progress: 950/2095 files checked...
Progress: 1000/2095 files checked...
Progress: 1050/2095 files checked...
Progress: 1100/2095 files chec

## 6. Display Empty Files Details

In [ ]:
if empty_files:
    print("Empty/Corrupted Files Found:")
    print("=" * 80)
    for idx, file_info in enumerate(empty_files, 1):
        print(f"{idx}. {file_info['folder']}/{file_info['filename']}")
        print(f"   Reason: {file_info['reason']}")
        print(f"   Size: {file_info['size']} bytes")
        print(f"   Full path: {file_info['path']}")
        print()
else:
    print("✓ No empty files found! All .nii files are valid.")

Empty/Corrupted Files Found:
1. BraTS-GLI-02124-101/BraTS-GLI-02124-101-seg.nii
   Reason: Error loading file: Expected 57768256 bytes, got 14991008 bytes from /media/basitali/C26423326423291D/zip data/BraTS-GLI-02124-101/BraTS-GLI-02124-101-seg.nii
 - could the file be damaged?
   Size: 14991360 bytes
   Full path: /media/basitali/C26423326423291D/zip data/BraTS-GLI-02124-101/BraTS-GLI-02124-101-seg.nii

2. BraTS-GLI-02124-101/BraTS-GLI-02124-101-t1c.nii
   Reason: File size is 0 bytes
   Size: 0 bytes
   Full path: /media/basitali/C26423326423291D/zip data/BraTS-GLI-02124-101/BraTS-GLI-02124-101-t1c.nii

3. BraTS-GLI-02124-101/BraTS-GLI-02124-101-t1n.nii
   Reason: File size is 0 bytes
   Size: 0 bytes
   Full path: /media/basitali/C26423326423291D/zip data/BraTS-GLI-02124-101/BraTS-GLI-02124-101-t1n.nii

4. BraTS-GLI-02124-101/BraTS-GLI-02124-101-t2f.nii
   Reason: File size is 0 bytes
   Size: 0 bytes
   Full path: /media/basitali/C26423326423291D/zip data/BraTS-GLI-02124-101/BraTS

## 7. Remove Empty .nii Files

**⚠️ WARNING:** This will permanently delete empty files. Make sure you have backups if needed!

In [ ]:
# Remove empty files
removed_files = []
failed_removals = []

if empty_files:
    print(f"Removing {len(empty_files)} empty files...\n")
    print("=" * 80)
    
    for idx, file_info in enumerate(empty_files, 1):
        try:
            os.remove(file_info['path'])
            removed_files.append(file_info)
            print(f"[{idx}/{len(empty_files)}] ✓ REMOVED: {file_info['folder']}/{file_info['filename']}")
        except Exception as e:
            failed_removals.append({'file': file_info, 'error': str(e)})
            print(f"[{idx}/{len(empty_files)}] ✗ FAILED: {file_info['folder']}/{file_info['filename']}")
            print(f"         Error: {str(e)}")
    
    print("=" * 80)
    print(f"\nRemoval Summary:")
    print(f"  Successfully removed: {len(removed_files)}")
    print(f"  Failed to remove: {len(failed_removals)}")
    
    if failed_removals:
        print("\nFailed removals:")
        for item in failed_removals:
            print(f"  - {item['file']['path']}")
            print(f"    Error: {item['error']}")
else:
    print("No empty files to remove!")

Removing 690 empty files...

[1/690] ✓ REMOVED: BraTS-GLI-02124-101/BraTS-GLI-02124-101-seg.nii
[2/690] ✓ REMOVED: BraTS-GLI-02124-101/BraTS-GLI-02124-101-t1c.nii
[3/690] ✓ REMOVED: BraTS-GLI-02124-101/BraTS-GLI-02124-101-t1n.nii
[4/690] ✓ REMOVED: BraTS-GLI-02124-101/BraTS-GLI-02124-101-t2f.nii
[5/690] ✓ REMOVED: BraTS-GLI-02124-101/BraTS-GLI-02124-101-t2w.nii
[6/690] ✓ REMOVED: BraTS-GLI-02128-100/BraTS-GLI-02128-100-seg.nii
[7/690] ✓ REMOVED: BraTS-GLI-02128-100/BraTS-GLI-02128-100-t1c.nii
[8/690] ✓ REMOVED: BraTS-GLI-02128-100/BraTS-GLI-02128-100-t1n.nii
[9/690] ✓ REMOVED: BraTS-GLI-02128-100/BraTS-GLI-02128-100-t2f.nii
[10/690] ✓ REMOVED: BraTS-GLI-02128-100/BraTS-GLI-02128-100-t2w.nii
[11/690] ✓ REMOVED: BraTS-GLI-02128-101/BraTS-GLI-02128-101-seg.nii
[12/690] ✓ REMOVED: BraTS-GLI-02128-101/BraTS-GLI-02128-101-t1c.nii
[13/690] ✓ REMOVED: BraTS-GLI-02128-101/BraTS-GLI-02128-101-t1n.nii
[14/690] ✓ REMOVED: BraTS-GLI-02128-101/BraTS-GLI-02128-101-t2f.nii
[15/690] ✓ REMOVED: BraTS-GL

## 8. Generate Summary Report by Patient Folder

In [ ]:
# Group files by patient folder
folder_stats = defaultdict(lambda: {'total': 0, 'empty': 0, 'valid': 0, 'removed': 0})

for file_info in file_status:
    folder = file_info['folder']
    folder_stats[folder]['total'] += 1
    if file_info['is_empty']:
        folder_stats[folder]['empty'] += 1
    else:
        folder_stats[folder]['valid'] += 1

for removed_file in removed_files:
    folder = removed_file['folder']
    folder_stats[folder]['removed'] += 1

# Display summary
print("=" * 80)
print("SUMMARY REPORT BY PATIENT FOLDER")
print("=" * 80)
print(f"{'Patient Folder':<30} {'Total':<8} {'Valid':<8} {'Empty':<8} {'Removed':<8}")
print("-" * 80)

folders_with_issues = []
for folder in sorted(folder_stats.keys()):
    stats = folder_stats[folder]
    print(f"{folder:<30} {stats['total']:<8} {stats['valid']:<8} {stats['empty']:<8} {stats['removed']:<8}")
    if stats['empty'] > 0:
        folders_with_issues.append(folder)

print("=" * 80)
print(f"\nOverall Summary:")
print(f"  Total patient folders scanned: {len(folder_stats)}")
print(f"  Folders with empty files: {len(folders_with_issues)}")
print(f"  Total files scanned: {len(file_status)}")
print(f"  Valid files: {len(valid_files)}")
print(f"  Empty files found: {len(empty_files)}")
print(f"  Files removed: {len(removed_files)}")

if folders_with_issues:
    print(f"\nFolders that had empty files:")
    for folder in folders_with_issues:
        print(f"  - {folder}")

SUMMARY REPORT BY PATIENT FOLDER
Patient Folder                 Total    Valid    Empty    Removed 
--------------------------------------------------------------------------------
BraTS-GLI-00005-100            5        5        0        0       
BraTS-GLI-00005-101            5        5        0        0       
BraTS-GLI-00006-100            5        5        0        0       
BraTS-GLI-00006-101            5        5        0        0       
BraTS-GLI-00008-100            5        5        0        0       
BraTS-GLI-00008-101            5        5        0        0       
BraTS-GLI-00008-102            5        5        0        0       
BraTS-GLI-00008-103            5        5        0        0       
BraTS-GLI-00009-100            5        5        0        0       
BraTS-GLI-00009-101            5        5        0        0       
BraTS-GLI-00020-100            5        5        0        0       
BraTS-GLI-00020-101            5        5        0        0       
BraTS-GLI-00027

## 9. Save Report to File (Optional)

In [ ]:
# Save detailed report to a text file
report_filename = f"empty_nii_files_report_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"
report_path = os.path.join(ROOT_DIR, report_filename)

with open(report_path, 'w') as f:
    f.write("=" * 80 + "\n")
    f.write("BraTS Dataset - Empty .nii Files Report\n")
    f.write(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write("=" * 80 + "\n\n")
    
    f.write(f"Overall Summary:\n")
    f.write(f"  Total patient folders scanned: {len(folder_stats)}\n")
    f.write(f"  Total files scanned: {len(file_status)}\n")
    f.write(f"  Valid files: {len(valid_files)}\n")
    f.write(f"  Empty files found: {len(empty_files)}\n")
    f.write(f"  Files removed: {len(removed_files)}\n\n")
    
    if empty_files:
        f.write("=" * 80 + "\n")
        f.write("Empty Files Details:\n")
        f.write("=" * 80 + "\n")
        for idx, file_info in enumerate(empty_files, 1):
            f.write(f"\n{idx}. {file_info['folder']}/{file_info['filename']}\n")
            f.write(f"   Reason: {file_info['reason']}\n")
            f.write(f"   Size: {file_info['size']} bytes\n")
            f.write(f"   Full path: {file_info['path']}\n")
            f.write(f"   Status: {'REMOVED' if file_info in removed_files else 'NOT REMOVED'}\n")
    
    f.write("\n" + "=" * 80 + "\n")
    f.write("Summary by Patient Folder:\n")
    f.write("=" * 80 + "\n")
    f.write(f"{'Patient Folder':<30} {'Total':<8} {'Valid':<8} {'Empty':<8} {'Removed':<8}\n")
    f.write("-" * 80 + "\n")
    
    for folder in sorted(folder_stats.keys()):
        stats = folder_stats[folder]
        f.write(f"{folder:<30} {stats['total']:<8} {stats['valid']:<8} {stats['empty']:<8} {stats['removed']:<8}\n")

print(f"✓ Report saved to: {report_path}")

✓ Report saved to: /media/basitali/C26423326423291D/zip data/empty_nii_files_report_20260111_183011.txt


## 10. Check for Empty Patient Folders

In [ ]:
# Check which patient folders are now empty (no files remaining)
empty_folders = []
non_empty_folders = []

print("Checking patient folders for emptiness...\n")
print("=" * 80)

for folder_path in patient_folders:
    # Get all files in the folder
    folder_contents = os.listdir(folder_path)
    
    # Filter out hidden files and directories
    visible_files = [f for f in folder_contents if not f.startswith('.') and os.path.isfile(os.path.join(folder_path, f))]
    
    folder_name = os.path.basename(folder_path)
    
    if len(visible_files) == 0:
        empty_folders.append({
            'path': folder_path,
            'name': folder_name
        })
        print(f"EMPTY: {folder_name} (0 files)")
    else:
        non_empty_folders.append({
            'path': folder_path,
            'name': folder_name,
            'file_count': len(visible_files)
        })

print("=" * 80)
print(f"\nFolder Summary:")
print(f"  Total patient folders: {len(patient_folders)}")
print(f"  Non-empty folders: {len(non_empty_folders)}")
print(f"  Empty folders: {len(empty_folders)}")

if empty_folders:
    print(f"\nEmpty folders found:")
    for folder in empty_folders:
        print(f"  - {folder['name']}")

Checking patient folders for emptiness...

EMPTY: BraTS-GLI-02124-101 (0 files)
EMPTY: BraTS-GLI-02128-100 (0 files)
EMPTY: BraTS-GLI-02128-101 (0 files)
EMPTY: BraTS-GLI-02128-102 (0 files)
EMPTY: BraTS-GLI-02128-103 (0 files)
EMPTY: BraTS-GLI-02129-100 (0 files)
EMPTY: BraTS-GLI-02129-101 (0 files)
EMPTY: BraTS-GLI-02129-102 (0 files)
EMPTY: BraTS-GLI-02129-103 (0 files)
EMPTY: BraTS-GLI-02129-104 (0 files)
EMPTY: BraTS-GLI-02129-105 (0 files)
EMPTY: BraTS-GLI-02129-106 (0 files)
EMPTY: BraTS-GLI-02131-100 (0 files)
EMPTY: BraTS-GLI-02131-101 (0 files)
EMPTY: BraTS-GLI-02131-102 (0 files)
EMPTY: BraTS-GLI-02131-103 (0 files)
EMPTY: BraTS-GLI-02132-101 (0 files)
EMPTY: BraTS-GLI-02132-102 (0 files)
EMPTY: BraTS-GLI-02132-103 (0 files)
EMPTY: BraTS-GLI-02132-104 (0 files)
EMPTY: BraTS-GLI-02135-100 (0 files)
EMPTY: BraTS-GLI-02135-101 (0 files)
EMPTY: BraTS-GLI-02136-100 (0 files)
EMPTY: BraTS-GLI-02136-101 (0 files)
EMPTY: BraTS-GLI-02136-102 (0 files)
EMPTY: BraTS-GLI-02136-103 (0 fi

## 11. Remove Empty Patient Folders

**⚠️ WARNING:** This will permanently delete empty patient folders!

In [ ]:
# Remove empty patient folders
removed_folders = []
failed_folder_removals = []

if empty_folders:
    print(f"Removing {len(empty_folders)} empty patient folders...\n")
    print("=" * 80)
    
    for idx, folder_info in enumerate(empty_folders, 1):
        try:
            # Use os.rmdir for empty directories or shutil.rmtree for any directory
            import shutil
            shutil.rmtree(folder_info['path'])
            removed_folders.append(folder_info)
            print(f"[{idx}/{len(empty_folders)}] ✓ REMOVED: {folder_info['name']}")
        except Exception as e:
            failed_folder_removals.append({'folder': folder_info, 'error': str(e)})
            print(f"[{idx}/{len(empty_folders)}] ✗ FAILED: {folder_info['name']}")
            print(f"         Error: {str(e)}")
    
    print("=" * 80)
    print(f"\nFolder Removal Summary:")
    print(f"  Successfully removed: {len(removed_folders)}")
    print(f"  Failed to remove: {len(failed_folder_removals)}")
    
    if failed_folder_removals:
        print("\nFailed folder removals:")
        for item in failed_folder_removals:
            print(f"  - {item['folder']['path']}")
            print(f"    Error: {item['error']}")
else:
    print("No empty folders to remove!")

Removing 138 empty patient folders...

[1/138] ✓ REMOVED: BraTS-GLI-02124-101
[2/138] ✓ REMOVED: BraTS-GLI-02128-100
[3/138] ✓ REMOVED: BraTS-GLI-02128-101
[4/138] ✓ REMOVED: BraTS-GLI-02128-102
[5/138] ✓ REMOVED: BraTS-GLI-02128-103
[6/138] ✓ REMOVED: BraTS-GLI-02129-100
[7/138] ✓ REMOVED: BraTS-GLI-02129-101
[8/138] ✓ REMOVED: BraTS-GLI-02129-102
[9/138] ✓ REMOVED: BraTS-GLI-02129-103
[10/138] ✓ REMOVED: BraTS-GLI-02129-104
[11/138] ✓ REMOVED: BraTS-GLI-02129-105
[12/138] ✓ REMOVED: BraTS-GLI-02129-106
[13/138] ✓ REMOVED: BraTS-GLI-02131-100
[14/138] ✓ REMOVED: BraTS-GLI-02131-101
[15/138] ✓ REMOVED: BraTS-GLI-02131-102
[16/138] ✓ REMOVED: BraTS-GLI-02131-103
[17/138] ✓ REMOVED: BraTS-GLI-02132-101
[18/138] ✓ REMOVED: BraTS-GLI-02132-102
[19/138] ✓ REMOVED: BraTS-GLI-02132-103
[20/138] ✓ REMOVED: BraTS-GLI-02132-104
[21/138] ✓ REMOVED: BraTS-GLI-02135-100
[22/138] ✓ REMOVED: BraTS-GLI-02135-101
[23/138] ✓ REMOVED: BraTS-GLI-02136-100
[24/138] ✓ REMOVED: BraTS-GLI-02136-101
[25/138] ✓

## 12. Final Summary Report

In [ ]:
print("=" * 80)
print("FINAL CLEANUP SUMMARY")
print("=" * 80)
print(f"\n📁 PATIENT FOLDERS:")
print(f"  Total patient folders scanned: {len(patient_folders)}")
print(f"  Empty folders found: {len(empty_folders)}")
print(f"  Empty folders removed: {len(removed_folders)}")
print(f"  Remaining folders: {len(patient_folders) - len(removed_folders)}")

print(f"\n📄 .NII FILES:")
print(f"  Total .nii files scanned: {len(file_status)}")
print(f"  Valid .nii files: {len(valid_files)}")
print(f"  Empty/corrupted .nii files found: {len(empty_files)}")
print(f"  Empty .nii files removed: {len(removed_files)}")

print(f"\n✅ CLEANUP COMPLETED!")
print("=" * 80)

# Update the report file with folder removal info
if removed_folders:
    with open(report_path, 'a') as f:
        f.write("\n\n" + "=" * 80 + "\n")
        f.write("Empty Patient Folders Removed:\n")
        f.write("=" * 80 + "\n")
        for idx, folder_info in enumerate(removed_folders, 1):
            f.write(f"{idx}. {folder_info['name']}\n")
            f.write(f"   Path: {folder_info['path']}\n\n")
        
        f.write(f"\nTotal empty folders removed: {len(removed_folders)}\n")
    
    print(f"\n✓ Updated report with folder removal information: {report_path}")

FINAL CLEANUP SUMMARY

📁 PATIENT FOLDERS:
  Total patient folders scanned: 400
  Empty folders found: 138
  Empty folders removed: 138
  Remaining folders: 262

📄 .NII FILES:
  Total .nii files scanned: 1999
  Valid .nii files: 1309
  Empty/corrupted .nii files found: 690
  Empty .nii files removed: 690

✅ CLEANUP COMPLETED!

✓ Updated report with folder removal information: /media/basitali/C26423326423291D/zip data/empty_nii_files_report_20260111_183011.txt
